# Gold ao vivo — `gold_orders_live`

Mantém o **estado atual de cada pedido** a partir do feed de eventos de status da bronze, tolerando eventos duplicados e fora de ordem. Desenho: `docs/design/fase-5-streaming.md`; decisões: ADRs 0009 a 0012.

| Peça | Decisão |
|---|---|
| Execução | Structured Streaming com `foreachBatch`, gatilho `AvailableNow` (o serverless não suporta `ProcessingTime`) |
| Deduplicação | `dropDuplicatesWithinWatermark` sobre `(order_id, event_sequence)`, watermark de 4 dias contado pelo **instante de chegada à bronze** (`_ingested_at`), não pelo `event_ts` (ADR 0013) |
| Escrita | `MERGE` que só troca o status se `event_sequence` for **maior** que a gravada (idempotente) |
| Tabela | `olist.gold.gold_orders_live`, uma linha por pedido (tabela Delta própria, fora do pipeline `olist.gold`) |

**Parâmetro**

| Widget | Padrão | Uso |
|---|---|---|
| `run_id` | vazio | identificador da execução; no job recebe `{{job.run_id}}`, vazio gera um UUID |
| `teste_leitura` | `nao` | `nao` roda a escrita real em `gold_orders_live`; os demais valores rodam um teste e encerram o notebook: `leitura` e `robustez` (etapa 5.2), `merge`, `watermark` e `conferencia` (etapa 5.3), `risco` (etapa 5.4) |

**Partes** (o arquivo é entregue completo, mas cresce a cada parte)

1. Configuração, tabelas, leitura em streaming, redução do micro-lote e **teste da leitura em streaming** (etapas 5.1 e 5.2)
2. `foreachBatch` + `MERGE`, `ops.streaming_batches`, execução, testes e conferência (etapa 5.3, esta parte)
3. Regra de risco, recálculo a cada passada e lista de pedidos em risco (etapa 5.4, esta parte)
4. Validação independente: notebook `src/quality/06_validacao_live` (etapa 5.4)

In [ ]:
# O QUE FAZ: cria o parâmetro do notebook (caixa de texto no topo da página).
#
# FUNÇÃO:
# - dbutils.widgets.text(nome, valor_padrão, rótulo): cria um widget de texto; o valor é lido
#   depois com dbutils.widgets.get(nome) e sempre chega como string.
# - dbutils.widgets.dropdown(nome, valor_padrão, lista, rótulo): widget de lista suspensa (teste_leitura:
#   nao, leitura, robustez, merge, watermark ou conferencia; os testes só rodam quando o valor não é "nao").
dbutils.widgets.text("run_id", "", "ID da execução")
dbutils.widgets.dropdown("teste_leitura", "nao", ["nao", "leitura", "robustez", "merge", "watermark", "conferencia", "risco"], "Teste (etapas 5.2 e 5.3)")

In [ ]:
# O QUE FAZ: define as constantes do notebook: tabelas de origem e de destino, pasta do checkpoint,
#   tamanho do watermark, SLA e a configuração da regra de risco (ADR 0010, 0011 e 0012).
#   Todos os nomes derivam de CATALOG. Nada aqui lê ou grava dados.
#
# RECURSOS:
# - f"...{variavel}...": f-string, monta texto substituindo as variáveis entre chaves.
# - WATERMARK_DELAY: por quanto tempo, contado no instante de CHEGADA à bronze (_ingested_at), o Spark
#   lembra um evento já visto para reconhecer duplicados. O simulador atrasa eventos em até 3 dias e a
#   chegada só avança, então nenhum evento fica "mais velho que o watermark" (com event_ts como relógio,
#   o Spark descartaria o evento atrasado: ver o teste "watermark").
# - Checkpoint: pasta onde o Spark guarda até onde já leu e o estado da deduplicação. Apagar a
#   pasta faz o stream recomeçar do zero (o MERGE idempotente torna isso seguro, só mais caro).
# - /Volumes/<catálogo>/<schema>/<volume>: caminho de um Volume do Unity Catalog.
# - uuid.uuid4(): gera um identificador aleatório único; str() converte em texto.
# - "a or b": devolve "a" se tiver valor (texto não vazio); senão, "b".
# - str.strip(): remove espaços e quebras de linha do início e do fim do texto.
import uuid

CATALOG = "olist"

EVENTS_TABLE = f"{CATALOG}.bronze.order_status_events"
ORDERS_TABLE = f"{CATALOG}.bronze.orders"          # data estimada de entrega (etapa 5.3)
LIVE_TABLE = f"{CATALOG}.gold.gold_orders_live"
BATCHES_TABLE = f"{CATALOG}.ops.streaming_batches"  # um registro por micro-lote (etapa 5.3)

CHECKPOINT_DIR = f"/Volumes/{CATALOG}/ops/checkpoints/streaming/gold_orders_live"

WATERMARK_DELAY = "4 days"   # ADR 0010
SLA_MINUTES = 15             # ADR 0012

# Regra de risco (ADR 0011): ajustar aqui, não no código da regra. Usada na etapa 5.4.
RISK_CONFIG = {
    "em_risco_dias": 2,
    "status_abertos": ["created", "approved", "invoiced", "processing", "shipped"],
}

RUN_ID = dbutils.widgets.get("run_id").strip() or str(uuid.uuid4())

print(f"run_id:     {RUN_ID}")
print(f"origem:     {EVENTS_TABLE}")
print(f"destino:    {LIVE_TABLE}")
print(f"checkpoint: {CHECKPOINT_DIR}")
print(f"watermark:  {WATERMARK_DELAY} | SLA: {SLA_MINUTES} min")

## Tabelas

`gold_orders_live` guarda **uma linha por pedido**. As colunas de etapa (`created_at` a `delivered_at`) são preenchidas por **qualquer** evento, na ordem em que chegar; já o `status`, a `event_sequence` e o `event_ts` só avançam quando chega um evento com sequência maior. Por isso um `delivered` que chega antes do `shipped` não perde o instante do despacho quando o `shipped` aparecer depois.

`ops.streaming_batches` registra cada micro-lote (volume, eventos ignorados e latência), base dos critérios de aceite de latência e de eventos tardios.

In [ ]:
# O QUE FAZ: declara a função que cria as duas tabelas (estado e registro de micro-lotes) e a executa
#   para as tabelas de produção. A função recebe os nomes para que os testes criem cópias de prova
#   com exatamente o mesmo formato. Pode ser executada várias vezes: não altera uma tabela que já existe.
#
# FUNÇÃO DECLARADA:
# - create_tables(live_table, batches_table): cria as duas tabelas se não existirem e acrescenta em
#   batches_table as colunas que a Parte 2 passou a gravar (new_orders e batch_seconds).
#
# RECURSOS SQL:
# - CREATE TABLE IF NOT EXISTS: cria só se a tabela não existir.
# - COMMENT '...': descrição da tabela ou da coluna, que aparece no Catalog Explorer.
# - TIMESTAMP: data/hora com fuso da sessão; BIGINT: inteiro de 64 bits; DOUBLE: decimal.
# - ALTER TABLE ... ADD COLUMNS: acrescenta colunas ao fim da tabela (as linhas antigas ficam nulas).
# - spark.table(nome).columns: lista de colunas da tabela, para só acrescentar o que falta.
def create_tables(live_table: str, batches_table: str) -> None:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {live_table} (
            order_id                 STRING    COMMENT 'Identificador do pedido (chave: uma linha por pedido).',
            status                   STRING    COMMENT 'Status atual: o do evento de maior event_sequence já aplicado.',
            event_sequence           INT       COMMENT 'Sequência do último evento aplicado; só aumenta.',
            event_ts                 TIMESTAMP COMMENT 'Instante (simulado) do último evento aplicado.',
            created_at               TIMESTAMP COMMENT 'Instante do evento created (compra).',
            approved_at              TIMESTAMP COMMENT 'Instante do evento approved.',
            shipped_at               TIMESTAMP COMMENT 'Instante do evento shipped (despacho).',
            delivered_at             TIMESTAMP COMMENT 'Instante do evento delivered (entrega ao cliente).',
            estimated_delivery_date  TIMESTAMP COMMENT 'Data estimada de entrega informada na compra.',
            risco_atraso             STRING    COMMENT 'normal, em_risco ou atrasado; só pedidos abertos (ADR 0011).',
            _updated_at              TIMESTAMP COMMENT 'Momento em que a linha foi gravada pela última vez.',
            _run_id                  STRING    COMMENT 'Execução que gravou a linha pela última vez.'
        )
        COMMENT 'Estado atual de cada pedido, mantido por streaming (MERGE por event_sequence). Visão operacional ao vivo: os KPIs oficiais continuam na gold batch (ADR 0012).'
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {batches_table} (
            run_id               STRING    COMMENT 'Execução do notebook (job.run_id ou UUID).',
            batch_id             BIGINT    COMMENT 'Número do micro-lote dentro do checkpoint.',
            processed_at         TIMESTAMP COMMENT 'Momento em que o micro-lote foi gravado.',
            events_in            BIGINT    COMMENT 'Eventos que chegaram ao micro-lote depois da deduplicação.',
            orders_touched       BIGINT    COMMENT 'Pedidos distintos no micro-lote.',
            status_updated       BIGINT    COMMENT 'Pedidos cujo status avançou (sequência maior), incluindo os pedidos novos.',
            ignored_old_sequence BIGINT    COMMENT 'Pedidos cujo evento tinha sequência menor ou igual à gravada.',
            late_events          BIGINT    COMMENT 'Eventos com event_ts mais antigo que o watermark (aplicados, nunca descartados).',
            max_event_ts         TIMESTAMP COMMENT 'Maior event_ts do micro-lote (data simulada corrente).',
            max_latency_seconds  DOUBLE    COMMENT 'Maior intervalo entre a gravação do arquivo na landing e a gravação do micro-lote em gold_orders_live.'
        )
        COMMENT 'Um registro por micro-lote do streaming gold_orders_live: volume, eventos ignorados, tardios e latência (critérios 5 e 7 da Fase 5).'
    """)

    existing = {c.lower() for c in spark.table(batches_table).columns}
    extras = [
        ("new_orders", "BIGINT", "Pedidos que ainda não existiam em gold_orders_live (inseridos neste micro-lote)."),
        ("batch_seconds", "DOUBLE", "Duração do micro-lote (redução, junção com orders e MERGE), em segundos."),
    ]
    missing = [(n, t, c) for n, t, c in extras if n not in existing]
    if missing:
        spark.sql(f"ALTER TABLE {batches_table} ADD COLUMNS (" + ", ".join(f"{n} {t} COMMENT '{c}'" for n, t, c in missing) + ")")


create_tables(LIVE_TABLE, BATCHES_TABLE)
print("Tabelas prontas:", LIVE_TABLE, "e", BATCHES_TABLE)

## Leitura em streaming e deduplicação

A origem é a **tabela bronze**, lida como stream: a bronze é Delta *append-only*, então cada linha nova (inclusive duplicatas e reenvios de arquivo) aparece como linha nova para o stream. O relógio do watermark é o **instante de chegada à bronze** (`_ingested_at`), que só avança: a medição da etapa 5.3 mostrou que, com o `event_ts` como relógio, o Spark **descarta** (nem chega ao `MERGE`) qualquer evento mais velho que o watermark, inclusive um evento distinto, o que quebraria o critério de que tardio é contado e aplicado (ADR 0013).

A leitura tem duas camadas (`read_events_clean`, sem deduplicação, e `read_events_stream`, deduplicada) e ignora commits que reescrevem a bronze (`skipChangeCommits`), para que uma exclusão de cliente não derrube o stream. O teste da etapa 5.2, mais abaixo, mede as duas.

Pontos de atenção:

- `dropDuplicatesWithinWatermark` lembra cada chave `(order_id, event_sequence)` só até o watermark passar. Um duplicado que chega **depois** disso não é mais reconhecido pela deduplicação; quem o impede de corromper o estado é o `MERGE` por sequência (segunda defesa, ADR 0010).
- Medido na etapa 5.3 (teste `watermark`): com `_ingested_at` como relógio, o evento distinto atrasado é aplicado e contado, e o duplicado tardio é removido pela deduplicação ou, se ela já tiver esquecido a chave, ignorado pelo `MERGE`.

In [ ]:
# O QUE FAZ: declara a leitura do feed de eventos da bronze em três camadas pequenas: limpeza
#   (função pura), leitura em streaming sem deduplicação e leitura em streaming deduplicada. A
#   separação deixa medir, na etapa 5.2, quantas linhas a bronze entrega e quantas sobram depois da
#   deduplicação.
#
# FUNÇÕES DECLARADAS:
# - clean_events(df): função pura (DataFrame -> DataFrame, serve para stream e para tabela): descarta a
#   linha corrompida e o pedido sem id, tipa event_sequence (int) e event_ts (timestamp) e mantém as
#   colunas de auditoria da bronze.
# - read_events_clean(table, skip_change_commits): lê a tabela como stream e aplica clean_events. Com
#   skip_change_commits = True (padrão), o stream ignora commits que alteram ou apagam linhas já
#   existentes (UPDATE, DELETE, MERGE) e só lê as linhas acrescentadas. Sem isso, qualquer DELETE na
#   bronze (como a exclusão de cliente da Fase 6) derruba o stream.
# - read_events_stream(table, clock): read_events_clean + watermark + deduplicação por (order_id, event_sequence).
#   clock é a coluna que serve de relógio do watermark: _ingested_at (padrão: instante de chegada à
#   bronze, que só avança e por isso nunca deixa um evento "atrasado demais") ou event_ts (só para o
#   teste de comparação: o Spark descarta tudo o que é mais velho que o watermark).
#   Não executa nada: um stream só começa a rodar quando é gravado com writeStream.start().
#
# FUNÇÕES NATIVAS:
# - spark.readStream.table(tabela): lê uma tabela Delta como fonte de streaming; cada linha
#   acrescentada à tabela vira linha nova no stream.
# - .option("skipChangeCommits", "true"): opção da fonte Delta que ignora commits que reescrevem
#   dados existentes.
# - df.where(condição): filtra linhas. Column.isNull() / isNotNull(): verifica nulo.
# - "&" entre colunas: E lógico do Spark (cada lado entre parênteses, por precedência).
# - Column.cast("int"): converte o texto da bronze em inteiro.
# - F.to_timestamp(coluna, formato): texto -> TIMESTAMP. Usamos TIMESTAMP (e não TIMESTAMP_NTZ)
#   porque é o tipo de relógio mais usado pelo watermark.
# - df.withWatermark(coluna, atraso): declara a coluna de tempo (o relógio) e quanto atraso tolerar;
#   o Spark descarta o estado de deduplicação mais antigo que (maior valor visto - atraso) e DESCARTA os
#   eventos mais velhos que isso.
# - df.dropDuplicatesWithinWatermark(colunas): mantém uma linha por combinação das colunas dentro
#   do watermark. Requer o withWatermark antes.
from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

TS_FORMAT = "yyyy-MM-dd HH:mm:ss"


def clean_events(df: DataFrame) -> DataFrame:
    """Descarta lixo (linha corrompida, pedido sem id) e tipa as colunas. Vale para stream e para tabela."""
    return (
        df
        .where(F.col("_corrupt_record").isNull() & F.col("order_id").isNotNull())
        .select(
            "order_id",
            "status",
            F.col("event_sequence").cast("int").alias("event_sequence"),
            F.to_timestamp(F.col("event_ts"), TS_FORMAT).alias("event_ts"),
            "_ingested_at",
            "_source_file_modified_at",
        )
    )


def read_events_clean(table: str = EVENTS_TABLE, skip_change_commits: bool = True) -> DataFrame:
    """Eventos de status da bronze como stream, limpos e tipados (sem deduplicação)."""
    reader = spark.readStream
    if skip_change_commits:
        reader = reader.option("skipChangeCommits", "true")
    return clean_events(reader.table(table))


def read_events_stream(table: str = EVENTS_TABLE, clock: str = "_ingested_at") -> DataFrame:
    """Eventos de status da bronze, tipados e deduplicados dentro do watermark (stream)."""
    return (
        read_events_clean(table)
        .withWatermark(clock, WATERMARK_DELAY)
        .dropDuplicatesWithinWatermark(["order_id", "event_sequence"])
    )

In [ ]:
# O QUE FAZ: confere que a leitura é de fato um stream e mostra o schema. Não grava nada.
#
# RECURSOS:
# - df.isStreaming: True se o DataFrame é uma consulta em streaming.
# - df.printSchema(): lista as colunas e os tipos.
preview = read_events_stream()

print(f"isStreaming: {preview.isStreaming}")
preview.printSchema()

## Redução do micro-lote

Um micro-lote pode trazer **vários eventos do mesmo pedido**, em qualquer ordem. Antes do `MERGE`, o lote é reduzido a **uma linha por pedido**: o evento de maior sequência (status atual) e o instante de cada etapa. É uma função pura (DataFrame → DataFrame), testável sem stream.

In [ ]:
# O QUE FAZ: declara a função pura que reduz um micro-lote a uma linha por pedido, e o mapa de
#   status para a coluna de instante da etapa.
#
# FUNÇÃO DECLARADA:
# - reduce_batch(events): devolve uma linha por order_id com: status, event_sequence e event_ts
#   do evento de MAIOR sequência (desempate pelo event_ts mais recente) e created_at, approved_at,
#   shipped_at e delivered_at (o instante de cada etapa que aparece no lote, nulo se não apareceu).
#
# FUNÇÕES NATIVAS:
# - Window.partitionBy(chave).orderBy(...): janela que separa as linhas por pedido e as ordena.
#   F.col(...).desc(): ordem decrescente.
# - F.row_number().over(janela): numera as linhas de cada pedido a partir de 1; a linha 1 é o evento
#   de maior sequência.
# - df.groupBy(chave).agg(...): agrega por pedido.
# - F.max(F.when(condição, valor)): maior valor entre as linhas que atendem à condição (when sem
#   otherwise devolve nulo nas demais, e max ignora nulos). Dá o instante de cada etapa.
# - df.join(outro, on=coluna, how="inner"): junta pelas colunas indicadas.
# - [... for ... in dict.items()]: list comprehension; desempacota chave e valor do dicionário.
STAGE_COLUMNS = {
    "created": "created_at",
    "approved": "approved_at",
    "shipped": "shipped_at",
    "delivered": "delivered_at",
}


def reduce_batch(events: DataFrame) -> DataFrame:
    """Uma linha por pedido: status do evento de maior sequência + instante de cada etapa do lote."""
    newest_first = Window.partitionBy("order_id").orderBy(F.col("event_sequence").desc(), F.col("event_ts").desc())
    latest = (
        events
        .withColumn("_rank", F.row_number().over(newest_first))
        .where(F.col("_rank") == 1)
        .select("order_id", "status", "event_sequence", "event_ts")
    )
    stages = events.groupBy("order_id").agg(*[
        F.max(F.when(F.col("status") == status, F.col("event_ts"))).alias(column)
        for status, column in STAGE_COLUMNS.items()
    ])
    return latest.join(stages, on="order_id", how="inner")

In [ ]:
# O QUE FAZ: testa reduce_batch com 5 eventos montados à mão, sem stream e sem gravar nada:
#   - pedido A: o delivered (seq 4) chega ANTES do shipped (seq 3) e há um duplicado do created;
#   - pedido B: só o created.
#   Esperado: A fica com status delivered e, mesmo assim, com o instante do shipped preenchido.
#
# FUNÇÕES NATIVAS:
# - spark.createDataFrame(lista_de_tuplas, schema_ddl): DataFrame pequeno a partir de dados Python.
# - datetime(ano, mês, dia, hora, minuto): data/hora do Python.
# - df.collect(): traz as linhas ao driver; aceitável aqui só porque o resultado tem 2 linhas.
# - assert condição, mensagem: interrompe com erro se a condição for falsa.
from datetime import datetime

teste = spark.createDataFrame(
    [
        ("A", "delivered", 4, datetime(2028, 8, 25, 10, 0)),   # chegou primeiro, apesar de ser o último passo
        ("A", "shipped",   3, datetime(2028, 8, 23, 9, 0)),
        ("A", "created",   1, datetime(2028, 8, 22, 8, 0)),
        ("A", "created",   1, datetime(2028, 8, 22, 8, 0)),    # duplicado
        ("B", "created",   1, datetime(2028, 8, 22, 12, 0)),
    ],
    "order_id string, status string, event_sequence int, event_ts timestamp",
)

reduzido = {row["order_id"]: row for row in reduce_batch(teste).collect()}

assert len(reduzido) == 2, "esperado: uma linha por pedido"
assert reduzido["A"]["status"] == "delivered" and reduzido["A"]["event_sequence"] == 4, "A deveria estar delivered (seq 4)"
assert reduzido["A"]["shipped_at"] is not None, "A deveria manter o instante do shipped"
assert reduzido["B"]["status"] == "created" and reduzido["B"]["shipped_at"] is None, "B deveria estar created, sem despacho"

display(reduce_batch(teste).orderBy("order_id"))
print("reduce_batch: OK")

## Escrita: `foreachBatch` + `MERGE` (etapa 5.3)

Cada micro-lote (já deduplicado) passa por quatro passos:

1. **Reduzir** a uma linha por pedido (`reduce_batch`) e trazer a data estimada de entrega de `bronze.orders`.
2. **Medir** o lote: eventos, pedidos, quantos avançam o status e quantos são ignorados por sequência antiga, e quantos eventos são **tardios** (mais velhos que o maior `event_ts` já gravado menos o watermark). Tardio é **contado e aplicado**, nunca descartado em silêncio.
3. **`MERGE`** em `gold_orders_live`: `status`, `event_sequence` e `event_ts` só trocam quando a sequência do lote é **maior**; os instantes das etapas e a data estimada só preenchem o que ainda é nulo. A linha só é reescrita se algo mudou, então reprocessar não altera a tabela (nem `_updated_at`).
4. **Registrar** o lote em `ops.streaming_batches`, com a latência entre a gravação do arquivo na landing e o fim do `MERGE`. A gravação do registro é idempotente por `(run_id, batch_id)`.

A função `run_live` recebe as tabelas e o checkpoint, então a mesma lógica roda na produção e nos testes (com tabelas de prova em `olist.ops.teste_5_3_*`).

In [ ]:
# O QUE FAZ: declara a escrita da etapa 5.3: junção com a data estimada, medição do micro-lote, MERGE
#   idempotente, registro em ops.streaming_batches e a função que roda o stream (AvailableNow).
#   Nada é lido ou gravado aqui: só declarações.
#
# FUNÇÕES DECLARADAS:
# - with_estimated_delivery(reduced, orders_table): acrescenta estimated_delivery_date (de bronze.orders,
#   que pode ter linhas repetidas por reenvio de arquivo: usa o maior valor por pedido) só para os
#   pedidos do lote. Pedido sem linha em orders fica com nulo e é preenchido numa execução futura.
# - batch_stats(batch_df, late_before): uma passada no lote que devolve eventos, pedidos distintos,
#   eventos tardios (event_ts < late_before), maior event_ts e o instante (em segundos) do arquivo mais
#   antigo do lote.
# - merge_counts(source, live_table): antes do MERGE, quantos pedidos são novos, quantos avançam o
#   status e quantos seriam ignorados (sequência menor ou igual à gravada).
# - merge_into_live(source, live_table, run_id): o MERGE (regras na célula de texto acima).
# - make_merge_handler(live_table, batches_table, orders_table, run_id): devolve a função que o
#   foreachBatch chama a cada micro-lote.
# - run_live(stream_df, live_table, batches_table, orders_table, checkpoint, run_id): roda o stream
#   com gatilho AvailableNow e espera o fim.
# - state_fingerprint(table): (linhas, pedidos distintos, soma de hashes do estado). Duas tabelas com
#   o mesmo estado têm o mesmo resultado; ignora as colunas de auditoria (_updated_at e _run_id).
#
# FUNÇÕES NATIVAS:
# - DeltaTable.forName(spark, tabela).alias("t").merge(origem, condição): MERGE pela API Python.
#   .whenMatchedUpdate(condition, set): atualiza as linhas que já existem E atendem à condição;
#   as demais não são tocadas. .whenNotMatchedInsert(values): insere os pedidos novos. .execute() roda.
# - F.expr("..."): coluna a partir de um texto SQL (CASE WHEN, coalesce).
# - F.coalesce(a, b): primeiro valor não nulo. F.count_if(cond): linhas em que a condição vale.
# - F.countDistinct, F.max, F.min, F.unix_timestamp: agregações e conversão em segundos.
# - F.xxhash64(colunas): hash de 64 bits da linha; F.pmod(a, n): resto positivo (mantém a soma dos
#   hashes pequena para não estourar o bigint no modo ANSI do serverless).
# - timedelta(days=n): intervalo de n dias. time.time(): relógio real em segundos.
# - .option("txnAppId", ...) e .option("txnVersion", ...): escrita Delta idempotente; repetir a mesma
#   versão com o mesmo appId não grava duas vezes (o appId é o run_id, para um checkpoint novo não
#   ser confundido com o antigo).
# - spark.createDataFrame(lista, schema_ddl): DataFrame de uma linha com o registro do lote.
import time
from datetime import timedelta
from delta.tables import DeltaTable

WATERMARK_DAYS = int(WATERMARK_DELAY.split()[0])
STAGE_FIELDS = ["created_at", "approved_at", "shipped_at", "delivered_at"]
STATE_FIELDS = ["order_id", "status", "event_sequence", "event_ts", *STAGE_FIELDS, "estimated_delivery_date"]
BATCH_LOG_SCHEMA = (
    "run_id string, batch_id bigint, events_in bigint, orders_touched bigint, status_updated bigint, "
    "ignored_old_sequence bigint, late_events bigint, max_event_ts timestamp, max_latency_seconds double, "
    "new_orders bigint, batch_seconds double"
)


def with_estimated_delivery(reduced: DataFrame, orders_table: str) -> DataFrame:
    """Acrescenta a data estimada de entrega (de bronze.orders) aos pedidos do lote."""
    estimated = (
        spark.table(orders_table)
        .where(F.col("_corrupt_record").isNull() & F.col("order_id").isNotNull())
        .select("order_id", F.to_timestamp(F.col("order_estimated_delivery_date"), TS_FORMAT).alias("estimated_delivery_date"))
        .join(reduced.select("order_id"), on="order_id", how="left_semi")
        .groupBy("order_id")
        .agg(F.max("estimated_delivery_date").alias("estimated_delivery_date"))
    )
    return reduced.join(estimated, on="order_id", how="left")


def batch_stats(batch_df: DataFrame, late_before) -> dict:
    """Medidas do micro-lote, numa só passada."""
    late = (F.col("event_ts") < F.lit(late_before)) if late_before is not None else F.lit(False)
    return batch_df.agg(
        F.count("*").alias("events_in"),
        F.countDistinct("order_id").alias("orders_touched"),
        F.count_if(late).alias("late_events"),
        F.max("event_ts").alias("max_event_ts"),
        F.min(F.unix_timestamp("_source_file_modified_at")).alias("oldest_file_epoch"),
    ).first().asDict()


def merge_counts(source: DataFrame, live_table: str) -> dict:
    """Quantos pedidos do lote são novos, avançam o status ou seriam ignorados (antes do MERGE)."""
    current = spark.table(live_table).select("order_id", F.col("event_sequence").alias("current_sequence"))
    return (
        source.join(current, on="order_id", how="left")
        .agg(
            F.count_if(F.col("current_sequence").isNull()).alias("new_orders"),
            F.count_if(F.col("current_sequence").isNotNull() & (F.col("event_sequence") > F.col("current_sequence"))).alias("advanced"),
            F.count_if(F.col("current_sequence").isNotNull() & (F.col("event_sequence") <= F.col("current_sequence"))).alias("ignored"),
        )
        .first().asDict()
    )


def merge_into_live(source: DataFrame, live_table: str, run_id: str) -> None:
    """MERGE idempotente: status só avança com sequência maior; etapas e data estimada só preenchem nulos."""
    newer = "s.event_sequence > t.event_sequence"
    fills = [*STAGE_FIELDS, "estimated_delivery_date"]
    gains = " OR ".join(f"(t.{c} IS NULL AND s.{c} IS NOT NULL)" for c in fills)
    updates = {
        "status": F.expr(f"CASE WHEN {newer} THEN s.status ELSE t.status END"),
        "event_sequence": F.expr(f"CASE WHEN {newer} THEN s.event_sequence ELSE t.event_sequence END"),
        "event_ts": F.expr(f"CASE WHEN {newer} THEN s.event_ts ELSE t.event_ts END"),
        **{c: F.expr(f"coalesce(t.{c}, s.{c})") for c in fills},
        "_updated_at": F.current_timestamp(),
        "_run_id": F.lit(run_id),
    }
    inserts = {
        **{c: F.col(f"s.{c}") for c in STATE_FIELDS},
        "risco_atraso": F.lit(None).cast("string"),
        "_updated_at": F.current_timestamp(),
        "_run_id": F.lit(run_id),
    }
    (
        DeltaTable.forName(spark, live_table).alias("t")
        .merge(source.alias("s"), "t.order_id = s.order_id")
        .whenMatchedUpdate(condition=f"{newer} OR {gains}", set=updates)
        .whenNotMatchedInsert(values=inserts)
        .execute()
    )


def make_merge_handler(live_table: str, batches_table: str, orders_table: str, run_id: str):
    """Função chamada pelo foreachBatch a cada micro-lote."""
    def handler(batch_df: DataFrame, batch_id: int) -> None:
        started = time.time()
        live_max = spark.table(live_table).agg(F.max("event_ts")).first()[0]
        late_before = live_max - timedelta(days=WATERMARK_DAYS) if live_max is not None else None

        stats = batch_stats(batch_df, late_before)
        if stats["events_in"] == 0:
            return  # lote vazio (o Spark roda um extra para avançar o watermark): nada a gravar

        source = with_estimated_delivery(reduce_batch(batch_df), orders_table)
        counts = merge_counts(source, live_table)
        merge_into_live(source, live_table, run_id)
        finished = time.time()

        log = spark.createDataFrame(
            [(
                run_id, batch_id, stats["events_in"], stats["orders_touched"],
                counts["new_orders"] + counts["advanced"], counts["ignored"], stats["late_events"],
                stats["max_event_ts"], float(finished - stats["oldest_file_epoch"]),
                counts["new_orders"], float(finished - started),
            )],
            BATCH_LOG_SCHEMA,
        ).select(
            "run_id", "batch_id", F.current_timestamp().alias("processed_at"), "events_in", "orders_touched",
            "status_updated", "ignored_old_sequence", "late_events", "max_event_ts", "max_latency_seconds",
            "new_orders", "batch_seconds",
        )
        (
            log.write.format("delta").mode("append")
            .option("txnAppId", run_id).option("txnVersion", batch_id)
            .saveAsTable(batches_table)
        )
    return handler


def run_live(stream_df: DataFrame, live_table: str, batches_table: str, orders_table: str, checkpoint: str, run_id: str) -> None:
    """Uma passada AvailableNow do stream até o fim, gravando em live_table."""
    query = (
        stream_df.writeStream
        .foreachBatch(make_merge_handler(live_table, batches_table, orders_table, run_id))
        .option("checkpointLocation", checkpoint)
        .queryName("gold_orders_live")
        .trigger(availableNow=True)
        .start()
    )
    query.awaitTermination()


def state_fingerprint(table: str) -> tuple[int, int, int]:
    """(linhas, pedidos distintos, soma dos hashes do estado), sem as colunas de auditoria."""
    row = spark.table(table).agg(
        F.count("*").alias("linhas"),
        F.countDistinct("order_id").alias("pedidos"),
        F.sum(F.pmod(F.xxhash64(*[F.col(c) for c in STATE_FIELDS]), F.lit(1000000007))).alias("hash"),
    ).first()
    return int(row["linhas"]), int(row["pedidos"]), int(row["hash"] or 0)

## Risco de atraso (etapa 5.4)

A regra do [ADR 0011](../docs/adr/0011-regra-de-pedido-em-risco.md) vale **só para pedidos abertos** (`created`, `approved`, `invoiced`, `processing`, `shipped`), em dias da **data simulada** (a maior `event_ts` já gravada, nunca `current_date()`):

| Nível | Regra |
|---|---|
| `atrasado` | a data estimada de entrega já passou |
| `em_risco` | faltam de 0 a **2 dias** para a data estimada e o pedido ainda **não foi despachado** (`shipped`) |
| `normal` | demais pedidos abertos |
| nulo | pedido fechado (`delivered`, `canceled`, `unavailable`): não há o que acionar |

N = 2 e a lista de status abertos ficam em `RISK_CONFIG`, no topo do notebook. Como "hoje" avança mesmo para pedidos sem evento novo, o risco de **todos os pedidos abertos** é recalculado ao fim de cada passada (`UPDATE` só nas linhas que mudam). A lista de pedidos em risco é a view `olist.gold.vw_pedidos_em_risco`.

In [ ]:
# O QUE FAZ: declara a regra de risco de atraso, o recálculo e a view da lista de pedidos em risco. Só
#   declarações: nada é lido nem gravado aqui.
#
# FUNÇÕES DECLARADAS:
# - risk_sql(today): o CASE SQL da regra (texto), com a data simulada, N e os status abertos vindos de
#   RISK_CONFIG. Um só texto serve para o teste, o UPDATE e a validação: a regra existe num lugar só.
# - with_risk(df, today): função pura (DataFrame -> DataFrame) que acrescenta a coluna risco_atraso.
#   Precisa das colunas status e estimated_delivery_date.
# - simulated_today(live_table): a data simulada corrente = a data do maior event_ts gravado (S8).
# - refresh_risk(live_table, run_id): recalcula risco_atraso de todos os pedidos e grava só o que mudou.
#   Devolve (linhas alteradas, contagem por nível).
# - create_risk_view(live_table, view_name): cria ou substitui a view com os pedidos atrasados e em risco.
#
# FUNÇÕES NATIVAS E SQL:
# - CASE WHEN ... THEN ... ELSE ... END: escolhe o valor pela primeira condição verdadeira.
# - status IN (...) / NOT IN (...): pertence ao conjunto. to_date(ts): corta a hora.
# - datediff(fim, início): dias entre as datas (positivo = fim depois do início).
# - DATE'AAAA-MM-DD': literal de data. a <=> b: igualdade que trata nulo = nulo como verdadeiro.
# - UPDATE tabela SET ... WHERE ...: altera só as linhas da condição; devolve o nº de linhas afetadas.
# - CREATE OR REPLACE VIEW v (coluna COMMENT '...') AS SELECT ...: view com comentário de coluna.
# - df.groupBy(...).count().collect(): contagem por grupo (poucas linhas: aceitável no driver).
from datetime import date

RISK_VIEW = f"{CATALOG}.gold.vw_pedidos_em_risco"


def risk_sql(today: date) -> str:
    """CASE SQL do nível de risco de um pedido, contra a data simulada `today`."""
    opened = ", ".join(f"'{s}'" for s in RISK_CONFIG["status_abertos"])
    n = int(RISK_CONFIG["em_risco_dias"])
    return (
        f"CASE "
        f"WHEN status NOT IN ({opened}) THEN NULL "
        f"WHEN estimated_delivery_date IS NULL THEN 'normal' "
        f"WHEN to_date(estimated_delivery_date) < DATE'{today}' THEN 'atrasado' "
        f"WHEN status <> 'shipped' AND datediff(to_date(estimated_delivery_date), DATE'{today}') <= {n} THEN 'em_risco' "
        f"ELSE 'normal' END"
    )


def with_risk(df: DataFrame, today: date) -> DataFrame:
    """Acrescenta risco_atraso (atrasado, em_risco, normal ou nulo para pedido fechado)."""
    return df.withColumn("risco_atraso", F.expr(risk_sql(today)))


def simulated_today(live_table: str) -> date | None:
    """Data simulada corrente: a data do maior event_ts já gravado em live_table."""
    value = spark.table(live_table).agg(F.max("event_ts")).first()[0]
    return None if value is None else value.date()


def refresh_risk(live_table: str, run_id: str) -> tuple[int, dict]:
    """Recalcula risco_atraso de todos os pedidos; reescreve só as linhas em que o valor mudou."""
    today = simulated_today(live_table)
    if today is None:
        return 0, {}
    case = risk_sql(today)
    changed = spark.sql(f"""
        UPDATE {live_table}
        SET risco_atraso = {case}, _updated_at = current_timestamp(), _run_id = '{run_id}'
        WHERE NOT (risco_atraso <=> ({case}))
    """).first()[0]
    counts = {r["risco_atraso"]: r["count"] for r in spark.table(live_table).groupBy("risco_atraso").count().collect()}
    return int(changed), counts


def create_risk_view(live_table: str, view_name: str) -> None:
    """Lista de pedidos atrasados e em risco, sempre atual (view sobre gold_orders_live)."""
    spark.sql(f"""
        CREATE OR REPLACE VIEW {view_name} (
            order_id                 COMMENT 'Identificador do pedido.',
            risco_atraso             COMMENT 'atrasado (data estimada já passou) ou em_risco (faltam até N dias e o pedido não foi despachado); regra do ADR 0011.',
            status                   COMMENT 'Status atual do pedido (maior event_sequence).',
            estimated_delivery_date  COMMENT 'Data estimada de entrega informada na compra.',
            dias_para_a_entrega      COMMENT 'Dias entre a data simulada e a data estimada; negativo = dias de atraso.',
            created_at               COMMENT 'Instante da compra (evento created).',
            shipped_at               COMMENT 'Instante do despacho (evento shipped); nulo se ainda não despachado.',
            ultimo_evento            COMMENT 'Instante do último evento aplicado ao pedido.',
            data_simulada            COMMENT 'Data simulada corrente, contra a qual o risco foi calculado (S8).'
        )
        COMMENT 'Pedidos abertos atrasados ou em risco de atraso, para a Logística agir antes da entrega. Visão operacional ao vivo sobre gold_orders_live; os KPIs oficiais continuam na gold batch (ADR 0012).'
        AS
        SELECT l.order_id, l.risco_atraso, l.status, l.estimated_delivery_date,
               datediff(to_date(l.estimated_delivery_date), h.hoje) AS dias_para_a_entrega,
               l.created_at, l.shipped_at, l.event_ts AS ultimo_evento, h.hoje AS data_simulada
        FROM {live_table} l
        CROSS JOIN (SELECT CAST(max(event_ts) AS DATE) AS hoje FROM {live_table}) h
        WHERE l.risco_atraso IN ('atrasado', 'em_risco')
    """)

## Teste da leitura em streaming (etapa 5.2)

Roda só com `teste_leitura` diferente de `nao` e **encerra o notebook** antes de qualquer escrita em `gold_orders_live`. Os resultados ficam em `olist.ops.teste_5_2_lotes` (um registro por micro-lote) e o checkpoint de prova em `/Volumes/olist/ops/checkpoints/streaming/teste_5_2/`.

- **`leitura`:** duas leituras em `AvailableNow`, cada uma com checkpoint próprio: `raw` (limpa, sem deduplicação) e `dedup` (a leitura real, com watermark). Cada execução lê **só as linhas novas** da bronze desde a anterior. Confere que `raw` somou exatamente as linhas limpas da bronze e compara `dedup` com as chaves distintas.
- **`robustez`:** numa **cópia** (`SHALLOW CLONE`) da bronze, faz `DELETE` e `INSERT` e confere que a leitura **com** `skipChangeCommits` ignora o `DELETE` e lê só o `INSERT` (sem a opção, o `DELETE` derruba a leitura: `DELTA_SOURCE_TABLE_IGNORE_CHANGES`, observado nas execuções de prova). A bronze real não é tocada.
- Mede, por micro-lote, os segundos entre a gravação do arquivo na landing e a entrada na bronze, e entre o arquivo e a leitura (para comparar com o SLA de 15 min).

In [ ]:
# O QUE FAZ: declara as peças do teste da leitura (tabela de resultados, tratador de micro-lote e
#   executor). Não lê nem grava dados: só declara. A tabela de resultados é criada só quando o teste
#   vai rodar.
#
# FUNÇÕES DECLARADAS:
# - make_stats_handler(name): devolve a função que o foreachBatch chama a cada micro-lote. Ela mede
#   o lote (linhas, pedidos, menor e maior event_ts e os dois tempos de latência) e acrescenta uma
#   linha em ops.teste_5_2_lotes.
# - run_read_test(name, stream_df, checkpoint): roda o stream uma vez com AvailableNow (lê o que
#   chegou desde o checkpoint e termina) e espera o fim.
# - read_totals(name): linhas e lotes já registrados para a leitura "name" (todas as execuções).
#
# FUNÇÕES NATIVAS:
# - df.writeStream.foreachBatch(função): entrega cada micro-lote como DataFrame comum à função.
# - .option("checkpointLocation", pasta): onde o stream guarda até onde já leu.
# - .trigger(availableNow=True): processa tudo o que chegou e termina (único gatilho do serverless).
# - query.awaitTermination(): espera o stream terminar; um erro do stream é lançado aqui.
# - F.unix_timestamp(coluna): segundos desde 1970; a diferença de dois deles são segundos.
# - F.current_timestamp(): horário real do cálculo. F.countDistinct: valores distintos.
# - df.agg(...).select(...): uma linha de totais do micro-lote.
TESTE_LEITURA = dbutils.widgets.get("teste_leitura")

TEST_STATS_TABLE = f"{CATALOG}.ops.teste_5_2_lotes"
TEST_CLONE_TABLE = f"{CATALOG}.ops.teste_5_2_clone"
TEST_CHECKPOINT_DIR = f"/Volumes/{CATALOG}/ops/checkpoints/streaming/teste_5_2"


def make_stats_handler(name: str):
    """Tratador de micro-lote: mede o lote e grava uma linha em ops.teste_5_2_lotes."""
    def handler(batch_df: DataFrame, batch_id: int) -> None:
        (
            batch_df.agg(
                F.count("*").alias("linhas"),
                F.countDistinct("order_id").alias("pedidos"),
                F.min("event_ts").alias("min_event_ts"),
                F.max("event_ts").alias("max_event_ts"),
                F.max(F.unix_timestamp("_ingested_at") - F.unix_timestamp("_source_file_modified_at")).alias("s_arquivo_ate_bronze"),
                F.max(F.unix_timestamp(F.current_timestamp()) - F.unix_timestamp("_source_file_modified_at")).alias("s_arquivo_ate_leitura"),
            )
            .select(
                F.lit(name).alias("leitura"),
                F.lit(RUN_ID).alias("run_id"),
                F.lit(batch_id).cast("bigint").alias("batch_id"),
                F.current_timestamp().alias("processed_at"),
                "linhas", "pedidos", "min_event_ts", "max_event_ts", "s_arquivo_ate_bronze", "s_arquivo_ate_leitura",
            )
            .write.mode("append").saveAsTable(TEST_STATS_TABLE)
        )
    return handler


def run_read_test(name: str, stream_df: DataFrame, checkpoint: str | None = None) -> None:
    """Uma passada AvailableNow do stream, com o tratador de medição, até terminar."""
    query = (
        stream_df.writeStream
        .foreachBatch(make_stats_handler(name))
        .option("checkpointLocation", checkpoint or f"{TEST_CHECKPOINT_DIR}/{name}")
        .trigger(availableNow=True)
        .start()
    )
    query.awaitTermination()


def read_totals(name: str) -> tuple[int, int]:
    """(linhas, lotes) já registrados para a leitura `name`, somando todas as execuções."""
    row = spark.table(TEST_STATS_TABLE).where(F.col("leitura") == name).agg(
        F.coalesce(F.sum("linhas"), F.lit(0)).alias("linhas"), F.count("*").alias("lotes")
    ).first()
    return int(row["linhas"]), int(row["lotes"])


if TESTE_LEITURA != "nao":
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {TEST_STATS_TABLE} (
            leitura               STRING    COMMENT 'Leitura de prova: raw (sem deduplicação), dedup (com watermark) ou robustez_*.',
            run_id                STRING    COMMENT 'Execução do notebook.',
            batch_id              BIGINT    COMMENT 'Número do micro-lote no checkpoint da leitura.',
            processed_at          TIMESTAMP COMMENT 'Momento em que o micro-lote foi medido (horário real).',
            linhas                BIGINT    COMMENT 'Linhas do micro-lote (depois da deduplicação, na leitura dedup).',
            pedidos               BIGINT    COMMENT 'Pedidos distintos no micro-lote.',
            min_event_ts          TIMESTAMP COMMENT 'Menor event_ts do micro-lote (tempo simulado).',
            max_event_ts          TIMESTAMP COMMENT 'Maior event_ts do micro-lote (tempo simulado).',
            s_arquivo_ate_bronze  BIGINT    COMMENT 'Maior intervalo, em segundos, entre a gravação do arquivo na landing e a entrada na bronze.',
            s_arquivo_ate_leitura BIGINT    COMMENT 'Maior intervalo, em segundos, entre a gravação do arquivo na landing e a leitura em streaming.'
        )
        COMMENT 'Medições dos testes da leitura em streaming (etapa 5.2): um registro por micro-lote. Pode ser apagada ao fim da fase.'
    """)
print(f"teste_leitura: {TESTE_LEITURA}")

In [ ]:
# O QUE FAZ: teste "leitura" (só roda com teste_leitura = leitura). Lê o feed da bronze duas vezes,
#   cada uma com checkpoint próprio e uma passada AvailableNow, e confere:
#   1. a leitura raw (sem deduplicação) somou EXATAMENTE as linhas limpas da bronze: cada linha foi
#      lida uma vez, e uma execução sem linhas novas lê 0;
#   2. a leitura dedup (a real, com watermark) somou quantas linhas, comparada com as chaves
#      distintas (order_id, event_sequence) da bronze. Diferença é achado para a etapa 5.3, não erro.
#   Mostra os micro-lotes desta execução com os tempos de latência.
#
# FUNÇÕES NATIVAS:
# - df.select(...).distinct().count(): quantidade de combinações distintas de colunas.
# - assert condição, mensagem: interrompe com erro se a condição for falsa.
# - display(df): mostra o resultado em tabela.
if TESTE_LEITURA == "leitura":
    lidas_antes = {nome: read_totals(nome) for nome in ("raw", "dedup")}

    run_read_test("raw", read_events_clean())
    run_read_test("dedup", read_events_stream())

    lidas_depois = {nome: read_totals(nome) for nome in ("raw", "dedup")}
    bronze_limpa = clean_events(spark.table(EVENTS_TABLE))
    total_bronze = bronze_limpa.count()
    chaves_distintas = bronze_limpa.select("order_id", "event_sequence").distinct().count()

    novas_raw = lidas_depois["raw"][0] - lidas_antes["raw"][0]
    novas_dedup = lidas_depois["dedup"][0] - lidas_antes["dedup"][0]
    print(f"raw:   {novas_raw} linhas novas nesta execução em {lidas_depois['raw'][1] - lidas_antes['raw'][1]} micro-lote(s); acumulado {lidas_depois['raw'][0]} de {total_bronze} na bronze")
    print(f"dedup: {novas_dedup} linhas novas nesta execução em {lidas_depois['dedup'][1] - lidas_antes['dedup'][1]} micro-lote(s); acumulado {lidas_depois['dedup'][0]} (chaves distintas na bronze: {chaves_distintas})")

    assert lidas_depois["raw"][0] == total_bronze, f"a leitura raw somou {lidas_depois['raw'][0]} linhas, a bronze tem {total_bronze}"
    print("OK  raw: cada linha limpa da bronze foi lida exatamente uma vez")
    if lidas_depois["dedup"][0] == chaves_distintas:
        print("OK  dedup: soma = chaves distintas da bronze")
    else:
        print(f"ACHADO  dedup: soma {lidas_depois['dedup'][0]} difere das chaves distintas {chaves_distintas} (diferença {lidas_depois['dedup'][0] - chaves_distintas}); investigar na etapa 5.3")

    display(spark.table(TEST_STATS_TABLE).where(F.col("run_id") == RUN_ID).orderBy("leitura", "batch_id"))

In [ ]:
# O QUE FAZ: teste "robustez" (só roda com teste_leitura = robustez): o que a leitura faz quando a
#   tabela de origem sofre UPDATE/DELETE, como acontecerá com a exclusão de cliente (Fase 6).
#   Tudo numa CÓPIA da bronze (SHALLOW CLONE: copia só os metadados, os arquivos de dados são
#   compartilhados), que é apagada no fim. Passos:
#   1. lê a cópia inteira; 2. faz DELETE de um pedido e INSERT de 3 linhas; 3. lê COM skipChangeCommits
#   no mesmo checkpoint (espera só as 3 linhas inseridas). Confere que a bronze real não mudou.
#   O caso SEM skipChangeCommits (a leitura falha com DELTA_SOURCE_TABLE_IGNORE_CHANGES) foi observado
#   em execuções de prova, mas não é afirmado aqui: o Databricks derruba o comando inteiro quando
#   qualquer stream termina com erro, mesmo tratado, e isso falharia o job.
#
# RECURSOS:
# - CREATE OR REPLACE TABLE ... SHALLOW CLONE origem: cópia leve da tabela Delta.
# - DELETE FROM / INSERT INTO ... SELECT: altera a cópia (o DELETE reescreve arquivos: é o "commit de
#   mudança" que a fonte de streaming não aceita por padrão).
# - try / finally: a cópia e o checkpoint de prova são apagados com ou sem erro.
# - dbutils.fs.rm(pasta, recurse=True): apaga a pasta do checkpoint de prova.
if TESTE_LEITURA == "robustez":
    bronze_antes = spark.table(EVENTS_TABLE).count()
    checkpoint = f"{TEST_CHECKPOINT_DIR}/robustez"
    limpa = "_corrupt_record IS NULL AND order_id IS NOT NULL"
    try:
        spark.sql(f"DELETE FROM {TEST_STATS_TABLE} WHERE leitura LIKE 'robustez%'")  # repetível: recomeça as medições
        spark.sql(f"CREATE OR REPLACE TABLE {TEST_CLONE_TABLE} SHALLOW CLONE {EVENTS_TABLE}")
        dbutils.fs.rm(checkpoint, recurse=True)

        run_read_test("robustez_1_leitura_inicial", read_events_clean(TEST_CLONE_TABLE, skip_change_commits=False), checkpoint)
        inicial = read_totals("robustez_1_leitura_inicial")[0]
        print(f"1. leitura inicial da cópia: {inicial} linhas")

        vitima = spark.table(TEST_CLONE_TABLE).where(limpa).select("order_id").first()["order_id"]
        spark.sql(f"DELETE FROM {TEST_CLONE_TABLE} WHERE order_id = '{vitima}'")
        spark.sql(f"INSERT INTO {TEST_CLONE_TABLE} SELECT * FROM {TEST_CLONE_TABLE} WHERE {limpa} LIMIT 3")
        print(f"2. DELETE do pedido {vitima} e INSERT de 3 linhas na cópia")

        run_read_test("robustez_2_com_opcao", read_events_clean(TEST_CLONE_TABLE, skip_change_commits=True), checkpoint)
        novas = read_totals("robustez_2_com_opcao")[0]
        assert novas == 3, f"COM skipChangeCommits esperado 3 linhas (as inseridas); lidas {novas}"
        print(f"3. COM skipChangeCommits: leu {novas} linhas (só o INSERT; o DELETE foi ignorado): OK")
    finally:
        spark.sql(f"DROP TABLE IF EXISTS {TEST_CLONE_TABLE}")
        dbutils.fs.rm(checkpoint, recurse=True)

    assert spark.table(EVENTS_TABLE).count() == bronze_antes, "a bronze real mudou"
    print(f"bronze real intacta ({bronze_antes} linhas); cópia e checkpoint de prova apagados")

## Testes da escrita (etapa 5.3)

- **`merge`:** em tabelas de prova (`olist.ops.teste_5_3_*`), 3 passadas de eventos de 3 pedidos com `delivered` chegando antes do `shipped`, duplicados dentro e entre os lotes. Confere o estado final, os números de cada micro-lote e que reprocessar tudo com **checkpoint novo** dá o mesmo estado (mesmo hash).
- **`watermark`:** mede o que o Spark faz com um evento **mais velho que o watermark** (distinto e duplicado). É a verificação que ficou pendente da etapa 5.2 e decide se o evento tardio precisa ser contado antes da deduplicação.
- **`conferencia`:** sobre a tabela real, depois da execução: uma linha por pedido, estado igual a um cálculo independente em SQL a partir da bronze, etapas e data estimada, replay com checkpoint novo (mesmo hash) e comparação com `fact_orders`.

As tabelas e os checkpoints de prova são apagados no fim de cada teste.

In [ ]:
# O QUE FAZ: declara os apoios dos testes da etapa 5.3: nomes das tabelas de prova, criação das tabelas
#   de eventos e de pedidos de prova, inclusão de eventos e limpeza. As tabelas de prova têm o mesmo
#   formato da bronze nas colunas que a leitura usa.
#
# FUNÇÕES DECLARADAS:
# - reset_test_tables(): apaga e recria as tabelas e os checkpoints de prova.
# - add_events(rows, files_at): acrescenta eventos (order_id, status, event_sequence, event_ts em texto)
#   à tabela de eventos de prova, como se um arquivo novo tivesse chegado agora.
# - add_orders(rows): acrescenta pedidos (order_id, data estimada em texto) à tabela de pedidos de prova.
# - drop_test_tables(): apaga as tabelas e os checkpoints de prova.
#
# FUNÇÕES NATIVAS:
# - spark.createDataFrame(lista, schema_ddl) e .write.mode("append").saveAsTable(tabela): inclui linhas.
# - F.current_timestamp(): instante da inclusão (vira _ingested_at e _source_file_modified_at).
# - dbutils.fs.rm(pasta, recurse=True): apaga os checkpoints de prova.
T53 = {name: f"{CATALOG}.ops.teste_5_3_{name}" for name in ("eventos", "orders", "live", "lotes", "live_replay", "lotes_replay")}
T53_CHECKPOINTS = f"{TEST_CHECKPOINT_DIR}/etapa_5_3"


def drop_test_tables() -> None:
    for table in T53.values():
        spark.sql(f"DROP TABLE IF EXISTS {table}")
    dbutils.fs.rm(T53_CHECKPOINTS, recurse=True)


def reset_test_tables() -> None:
    drop_test_tables()
    spark.sql(f"""
        CREATE TABLE {T53['eventos']} (
            order_id STRING, status STRING, event_sequence STRING, event_ts STRING, _corrupt_record STRING,
            _ingested_at TIMESTAMP, _source_file_modified_at TIMESTAMP
        ) COMMENT 'Prova da etapa 5.3: eventos de status no formato da bronze. Pode ser apagada.'
    """)
    spark.sql(f"""
        CREATE TABLE {T53['orders']} (order_id STRING, order_estimated_delivery_date STRING, _corrupt_record STRING)
        COMMENT 'Prova da etapa 5.3: pedidos no formato da bronze. Pode ser apagada.'
    """)
    create_tables(T53["live"], T53["lotes"])
    create_tables(T53["live_replay"], T53["lotes_replay"])


def add_events(rows: list[tuple[str, str, int, str]]) -> None:
    (
        spark.createDataFrame([(o, s, str(q), t) for o, s, q, t in rows], "order_id string, status string, event_sequence string, event_ts string")
        .withColumns({"_corrupt_record": F.lit(None).cast("string"), "_ingested_at": F.current_timestamp(), "_source_file_modified_at": F.current_timestamp()})
        .write.mode("append").saveAsTable(T53["eventos"])
    )


def add_orders(rows: list[tuple[str, str]]) -> None:
    (
        spark.createDataFrame(rows, "order_id string, order_estimated_delivery_date string")
        .withColumn("_corrupt_record", F.lit(None).cast("string"))
        .write.mode("append").saveAsTable(T53["orders"])
    )


def run_test_pass(name: str, live: str = None, lotes: str = None) -> None:
    """Uma passada do stream de prova (checkpoint próprio por nome)."""
    run_live(
        read_events_stream(T53["eventos"]), live or T53["live"], lotes or T53["lotes"], T53["orders"],
        f"{T53_CHECKPOINTS}/{name}", RUN_ID,
    )

In [ ]:
# O QUE FAZ: teste "merge" (só roda com teste_leitura = merge), em tabelas de prova.
#   Cenário (3 passadas, cada uma é um "arquivo novo"):
#   1. pedido A chega já com o delivered (seq 4); pedido B com o created;
#   2. chegam o shipped (seq 3) e o created de A (atrasados em relação ao delivered), o approved de B
#      duas vezes (duplicado no mesmo lote) e o created de C;
#   3. os mesmos eventos de A e B chegam de novo (reenvio de arquivo) e C é aprovado.
#   Confere: (a) estado final de cada pedido; (b) os números de cada micro-lote em ops.teste_5_3_lotes;
#   (c) uma linha por pedido; (d) reprocessar tudo com um checkpoint NOVO em outra tabela dá o mesmo
#   estado (mesmo hash): é a idempotência; (e) a data estimada de entrega veio de orders.
#
# FUNÇÕES NATIVAS:
# - df.collect(): traz as linhas ao driver (aqui, 3 linhas).
# - assert condição, mensagem: interrompe com erro se a condição for falsa.
# - display(df): mostra o resultado em tabela.
if TESTE_LEITURA == "merge":
    try:
        reset_test_tables()
        add_orders([("A", "2028-09-10 00:00:00"), ("B", "2028-09-12 00:00:00"), ("C", "2028-09-14 00:00:00")])

        add_events([("A", "delivered", 4, "2028-08-25 10:00:00"), ("B", "created", 1, "2028-08-22 12:00:00")])
        run_test_pass("principal")

        add_events([
            ("A", "shipped", 3, "2028-08-23 09:00:00"), ("A", "created", 1, "2028-08-22 08:00:00"),
            ("B", "approved", 2, "2028-08-22 13:00:00"), ("B", "approved", 2, "2028-08-22 13:00:00"),
            ("C", "created", 1, "2028-08-23 08:00:00"),
        ])
        run_test_pass("principal")

        add_events([
            ("A", "shipped", 3, "2028-08-23 09:00:00"), ("B", "approved", 2, "2028-08-22 13:00:00"),
            ("C", "approved", 2, "2028-08-23 09:00:00"),
        ])
        run_test_pass("principal")

        live = {r["order_id"]: r for r in spark.table(T53["live"]).collect()}
        assert set(live) == {"A", "B", "C"} and spark.table(T53["live"]).count() == 3, "esperado uma linha por pedido (A, B, C)"
        a, b, c = live["A"], live["B"], live["C"]
        assert (a["status"], a["event_sequence"]) == ("delivered", 4), f"A deveria estar delivered/4: {a['status']}/{a['event_sequence']}"
        assert a["created_at"] is not None and a["shipped_at"] is not None and a["delivered_at"] is not None, "A deveria ter os instantes de created, shipped e delivered"
        assert (b["status"], b["event_sequence"]) == ("approved", 2) and b["approved_at"] is not None, "B deveria estar approved/2"
        assert (c["status"], c["event_sequence"]) == ("approved", 2), "C deveria estar approved/2"
        assert str(a["estimated_delivery_date"]).startswith("2028-09-10"), "A deveria ter a data estimada de orders"
        print("OK  (a) estado final: A delivered com shipped_at preenchido; B e C approved; data estimada de orders")

        lotes = spark.table(T53["lotes"]).orderBy("processed_at").collect()
        medido = [(r["events_in"], r["status_updated"], r["ignored_old_sequence"], r["new_orders"]) for r in lotes]
        esperado = [(2, 2, 0, 2), (4, 2, 1, 1), (1, 1, 0, 0)]
        assert medido == esperado, f"micro-lotes (eventos, status avançou, ignorados, novos): medido {medido}, esperado {esperado}"
        print(f"OK  (b) micro-lotes: {medido}  (os duplicados dos reenvios foram removidos pela deduplicação: 1 evento no último lote)")

        antes = state_fingerprint(T53["live"])
        run_test_pass("replay", T53["live_replay"], T53["lotes_replay"])
        depois = state_fingerprint(T53["live_replay"])
        assert antes == depois, f"replay com checkpoint novo mudou o estado: {antes} x {depois}"
        print(f"OK  (c, d) uma linha por pedido; replay com checkpoint novo = mesmo estado (linhas, pedidos, hash) = {antes}")

        # reaplicar os mesmos eventos na tabela já pronta (checkpoint novo, mesma tabela) não pode mudar nada
        run_test_pass("replay_mesma_tabela")
        assert state_fingerprint(T53["live"]) == antes, "reaplicar na mesma tabela mudou o estado"
        print("OK  (d) reaplicar os eventos na mesma tabela, com checkpoint novo, não muda o estado")

        display(spark.table(T53["live"]).orderBy("order_id"))
        display(spark.table(T53["lotes"]).orderBy("processed_at"))
        print("teste merge: OK")
    finally:
        drop_test_tables()
        print("tabelas e checkpoints de prova apagados")

In [ ]:
# O QUE FAZ: teste "watermark" (só roda com teste_leitura = watermark), em tabelas de prova. Mede o que
#   a deduplicação faz com eventos mais velhos que o watermark (4 dias), com DOIS relógios de watermark
#   lidos da mesma tabela de eventos (cada um com checkpoint e tabela de estado próprios):
#   - "event_ts": o relógio do rascunho do ADR 0010 (tempo do evento);
#   - "_ingested_at": o relógio adotado (instante de chegada à bronze, ADR 0013).
#   Cenário: passada 1, eventos de W1 e W2 (20 e 21/08); passada 2, um evento de W3 em 30/08 (leva o
#   watermark do relógio event_ts a 26/08); passada 3, um evento NOVO de W1 (approved, 20/08) e os
#   duplicados do created de W1 e de W2, todos com event_ts mais velho que 26/08.
#   Afirma: nenhum pedido repetido e nenhum estado retrocedido em qualquer relógio; com o relógio
#   _ingested_at, o evento distinto atrasado é APLICADO (W1 fica approved/2) e CONTADO (late_events = 1)
#   e os duplicados não chegam ao MERGE. Com event_ts apenas registra o que acontece (medição).
#
# FUNÇÕES NATIVAS:
# - df.where(...).first(): primeira linha (ou None) do filtro.
# - dict e f-string: montam o resumo de cada relógio.
if TESTE_LEITURA == "watermark":
    relogios = {"event_ts": (T53["live"], T53["lotes"]), "_ingested_at": (T53["live_replay"], T53["lotes_replay"])}

    def run_both(label: str) -> None:
        for relogio, (live_t, lotes_t) in relogios.items():
            run_live(read_events_stream(T53["eventos"], relogio), live_t, lotes_t, T53["orders"], f"{T53_CHECKPOINTS}/{relogio}", RUN_ID)

    try:
        reset_test_tables()
        add_orders([("W1", "2028-09-10 00:00:00"), ("W2", "2028-09-11 00:00:00"), ("W3", "2028-09-12 00:00:00")])

        add_events([("W1", "created", 1, "2028-08-20 10:00:00"), ("W2", "created", 1, "2028-08-21 10:00:00")])
        run_both("passada 1")
        add_events([("W3", "created", 1, "2028-08-30 10:00:00")])
        run_both("passada 2")
        antes = {r: spark.table(lotes_t).count() for r, (_, lotes_t) in relogios.items()}

        add_events([
            ("W1", "approved", 2, "2028-08-20 12:00:00"),   # evento distinto, event_ts mais velho que o watermark
            ("W1", "created", 1, "2028-08-20 10:00:00"),    # duplicado tardio
            ("W2", "created", 1, "2028-08-21 10:00:00"),    # duplicado tardio
        ])
        run_both("passada 3")

        for relogio, (live_t, lotes_t) in relogios.items():
            assert spark.table(live_t).count() == 3, f"[{relogio}] pedido repetido em gold_orders_live"
            w1 = spark.table(live_t).where("order_id = 'W1'").first()
            assert w1["event_sequence"] >= 1 and w1["status"] in ("created", "approved"), f"[{relogio}] estado de W1 retrocedeu"
            novos = spark.table(lotes_t).orderBy("processed_at").collect()[antes[relogio]:]
            chegaram = sum(r["events_in"] for r in novos)
            tardios = sum(r["late_events"] for r in novos)
            ignorados = sum(r["ignored_old_sequence"] for r in novos)
            aplicado = w1["event_sequence"] == 2
            print(f"relógio {relogio:13} passada 3: chegaram ao MERGE {chegaram} evento(s), tardios contados {tardios}, ignorados por sequência {ignorados}; "
                  f"W1 final = {w1['status']}/{w1['event_sequence']} -> evento distinto atrasado {'APLICADO' if aplicado else 'DESCARTADO'}")
            if relogio == "_ingested_at":
                assert aplicado, "com o relógio _ingested_at o evento distinto atrasado deveria ser aplicado"
                assert tardios == 1, f"o evento atrasado deveria ser contado (late_events = 1); contado {tardios}"
                assert chegaram == 1, f"os duplicados deveriam ser removidos pela deduplicação; chegaram {chegaram} eventos ao MERGE"
        print("teste watermark: OK (relógio _ingested_at aplica e conta o tardio; relógio event_ts registrado só como medição)")
    finally:
        drop_test_tables()
        print("tabelas e checkpoints de prova apagados")

In [ ]:
# O QUE FAZ: conferência da etapa 5.3 sobre a tabela REAL (só roda com teste_leitura = conferencia, depois
#   da execução com teste_leitura = nao). Não altera gold_orders_live. Verificações (todas com assert):
#   1. uma linha por pedido e nenhum order_id repetido;
#   2. estado = cálculo independente em SQL: o status é o do evento de maior event_sequence entre os eventos
#      limpos e distintos da bronze (sem usar nenhuma função deste notebook);
#   3. os instantes de created, approved, shipped e delivered batem com a bronze;
#   4. a data estimada bate com a de bronze.orders;
#   5. replay: o mesmo stream lido do início com um CHECKPOINT NOVO em uma tabela de prova dá o mesmo
#      estado (mesmo hash) que a tabela real: é a idempotência da etapa;
#   6. pré-conferência com fact_orders: nos pedidos sem nenhum evento depois de 2028-08-21, o status é
#      o mesmo da gold batch (a validação formal, em todos os pedidos, é da etapa 5.4);
#   7. idempotência sem arquivo novo: roda o stream real mais uma vez; se nenhum micro-lote novo apareceu
#      (não havia arquivo novo), o estado tem que ser idêntico (mesmo hash). Se havia dado novo, o passo
#      só aplica e avisa (a comparação não se aplica).
#   No fim grava o resultado em ops.streaming_conferencias (o notebook de validação lê esse registro: o
#   Free Edition não permite rodar um notebook dentro de outro, então a validação não executa o monitor).
#
# RECURSOS SQL:
# - max_by(valor, ordem): o valor da linha em que "ordem" é a maior (aqui, o status do evento de maior
#   sequência). IS DISTINCT FROM: comparação que trata nulo como valor (nulo = nulo é "igual").
# - FULL OUTER JOIN: mantém os pedidos que existem só de um lado. count_if(cond): linhas da condição.
# - WITH nome AS (...): expressão de tabela comum, uma "tabela" temporária só desta consulta.
if TESTE_LEITURA == "conferencia":
    live = spark.table(LIVE_TABLE)
    total, distintos = live.count(), live.select("order_id").distinct().count()
    assert total == distintos, f"order_id repetido: {total} linhas para {distintos} pedidos"
    print(f"OK  1. {total} linhas, {distintos} pedidos distintos")

    base = f"""
        WITH ev AS (
            SELECT order_id, status, CAST(event_sequence AS INT) AS seq, to_timestamp(event_ts, 'yyyy-MM-dd HH:mm:ss') AS ts
            FROM {EVENTS_TABLE} WHERE _corrupt_record IS NULL AND order_id IS NOT NULL
        ),
        esperado AS (
            SELECT order_id,
                   max_by(status, seq) AS status, max(seq) AS seq,
                   max(CASE WHEN status = 'created'   THEN ts END) AS created_at,
                   max(CASE WHEN status = 'approved'  THEN ts END) AS approved_at,
                   max(CASE WHEN status = 'shipped'   THEN ts END) AS shipped_at,
                   max(CASE WHEN status = 'delivered' THEN ts END) AS delivered_at
            FROM ev GROUP BY order_id
        )
    """
    resultado = spark.sql(base + f"""
        SELECT count(*) AS pedidos_comparados,
               count_if(e.order_id IS NULL OR l.order_id IS NULL) AS so_de_um_lado,
               count_if(e.status IS DISTINCT FROM l.status OR e.seq IS DISTINCT FROM l.event_sequence) AS status_diferente,
               count_if(e.created_at IS DISTINCT FROM l.created_at OR e.approved_at IS DISTINCT FROM l.approved_at
                        OR e.shipped_at IS DISTINCT FROM l.shipped_at OR e.delivered_at IS DISTINCT FROM l.delivered_at) AS etapas_diferentes
        FROM esperado e FULL OUTER JOIN {LIVE_TABLE} l ON e.order_id = l.order_id
    """).first()
    assert resultado["so_de_um_lado"] == 0, f"pedidos só de um lado: {resultado['so_de_um_lado']}"
    assert resultado["status_diferente"] == 0, f"status ou sequência diferentes do cálculo independente: {resultado['status_diferente']}"
    print(f"OK  2. {resultado['pedidos_comparados']} pedidos: status e sequência iguais ao cálculo independente em SQL")
    assert resultado["etapas_diferentes"] == 0, f"instantes de etapa diferentes: {resultado['etapas_diferentes']}"
    print("OK  3. instantes de created, approved, shipped e delivered iguais aos da bronze")

    estimada = spark.sql(f"""
        WITH o AS (
            SELECT order_id, max(to_timestamp(order_estimated_delivery_date, 'yyyy-MM-dd HH:mm:ss')) AS estimada
            FROM {ORDERS_TABLE} WHERE _corrupt_record IS NULL AND order_id IS NOT NULL GROUP BY order_id
        )
        SELECT count_if(l.estimated_delivery_date IS DISTINCT FROM o.estimada) AS diferentes,
               count_if(l.estimated_delivery_date IS NULL) AS nulas
        FROM {LIVE_TABLE} l LEFT JOIN o ON l.order_id = o.order_id
    """).first()
    assert estimada["diferentes"] == 0, f"data estimada diferente de bronze.orders em {estimada['diferentes']} pedidos"
    print(f"OK  4. data estimada igual à de bronze.orders ({estimada['nulas']} pedidos sem data estimada)")

    try:
        reset_test_tables()
        real = state_fingerprint(LIVE_TABLE)
        run_live(read_events_stream(), T53["live_replay"], T53["lotes_replay"], ORDERS_TABLE, f"{T53_CHECKPOINTS}/replay_real", RUN_ID)
        replay = state_fingerprint(T53["live_replay"])
        assert real == replay, f"replay com checkpoint novo difere da tabela real: {real} x {replay}"
        print(f"OK  5. replay com checkpoint novo = tabela real (linhas, pedidos, hash) = {real}")
    finally:
        drop_test_tables()

    batch = spark.sql(f"""
        WITH apos AS (
            SELECT order_id, max(sim_date) AS ultimo_dia FROM {EVENTS_TABLE}
            WHERE _corrupt_record IS NULL AND order_id IS NOT NULL GROUP BY order_id
        )
        SELECT count(*) AS comparados, count_if(l.status != f.status_atual) AS diferentes
        FROM {LIVE_TABLE} l JOIN apos a ON l.order_id = a.order_id AND a.ultimo_dia <= '2028-08-21'
        JOIN {CATALOG}.gold.fact_orders f ON f.order_id = l.order_id
    """).first()
    assert batch["diferentes"] == 0, f"status diferente de fact_orders em {batch['diferentes']} de {batch['comparados']} pedidos"
    print(f"OK  6. {batch['comparados']} pedidos sem evento depois de 2028-08-21: status igual ao de fact_orders")

    lotes_antes = spark.table(BATCHES_TABLE).count()
    estado_antes = state_fingerprint(LIVE_TABLE)
    run_live(read_events_stream(), LIVE_TABLE, BATCHES_TABLE, ORDERS_TABLE, CHECKPOINT_DIR, RUN_ID)
    lotes_novos = spark.table(BATCHES_TABLE).count() - lotes_antes
    estado_depois = state_fingerprint(LIVE_TABLE)
    if lotes_novos == 0:
        assert estado_antes == estado_depois, f"rodar sem arquivo novo mudou o estado: {estado_antes} x {estado_depois}"
        sem_arquivo_novo_ok = True
        print(f"OK  7. rodar o stream real sem arquivo novo não grava micro-lote nem muda o estado {estado_depois}")
    else:
        sem_arquivo_novo_ok = None
        print(f"AVISO 7. havia {lotes_novos} micro-lote(s) novo(s): o estado foi atualizado e a comparação sem arquivo novo não se aplica; rode a conferência de novo")

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {CATALOG}.ops.streaming_conferencias (
            run_id                 STRING    COMMENT 'Execução do notebook que fez a conferência.',
            conferido_em           TIMESTAMP COMMENT 'Momento em que a conferência terminou (horário real).',
            linhas                 BIGINT    COMMENT 'Linhas de gold_orders_live no momento da conferência.',
            pedidos                BIGINT    COMMENT 'Pedidos distintos.',
            hash_estado            BIGINT    COMMENT 'Soma dos hashes do estado (sem colunas de auditoria).',
            replay_ok              BOOLEAN   COMMENT 'Replay com checkpoint novo deu o mesmo estado.',
            sem_arquivo_novo_ok    BOOLEAN   COMMENT 'Rodar sem arquivo novo não mudou o estado; nulo se havia dado novo.'
        )
        COMMENT 'Resultado das conferências do monitor ao vivo (etapa 5.4), lido pelo notebook de validação.'
    """)
    (
        spark.createDataFrame(
            [(RUN_ID, estado_depois[0], estado_depois[1], estado_depois[2], True, sem_arquivo_novo_ok)],
            "run_id string, linhas bigint, pedidos bigint, hash_estado bigint, replay_ok boolean, sem_arquivo_novo_ok boolean",
        )
        .select("run_id", F.current_timestamp().alias("conferido_em"), "linhas", "pedidos", "hash_estado", "replay_ok", "sem_arquivo_novo_ok")
        .write.mode("append").saveAsTable(f"{CATALOG}.ops.streaming_conferencias")
    )

    display(spark.table(BATCHES_TABLE).orderBy(F.col("processed_at").desc()).limit(20))
    print("conferência das etapas 5.3 e 5.4: OK")

In [ ]:
# O QUE FAZ: teste "risco" (só roda com teste_leitura = risco): confere a regra de risco com pedidos montados
#   à mão e a data simulada 2028-08-25, sem gravar nada. Casos (status, data estimada -> esperado):
#   shipped, 24/08 -> atrasado (a data passou, mesmo despachado); approved, 25/08 -> em_risco (0 dia);
#   approved, 27/08 -> em_risco (2 dias); approved, 28/08 -> normal (3 dias); shipped, 26/08 -> normal
#   (despachado e no prazo); processing, 24/08 -> atrasado; invoiced, 25/08 -> em_risco; created, sem data
#   estimada -> normal; delivered, canceled e unavailable -> nulo (pedido fechado, mesmo com data vencida).
#   Também confere que with_risk e o UPDATE usam o mesmo texto da regra.
#
# FUNÇÕES NATIVAS:
# - spark.createDataFrame(lista, schema_ddl): DataFrame pequeno a partir de dados Python.
# - to_timestamp('AAAA-MM-DD'): texto -> data/hora (cast("timestamp")). dict(...): mapa pedido -> esperado.
# - assert condição, mensagem: interrompe com erro se a condição for falsa.
if TESTE_LEITURA == "risco":
    hoje = date(2028, 8, 25)
    casos = [
        ("r1", "shipped", "2028-08-24", "atrasado"), ("r2", "approved", "2028-08-25", "em_risco"),
        ("r3", "approved", "2028-08-27", "em_risco"), ("r4", "approved", "2028-08-28", "normal"),
        ("r5", "shipped", "2028-08-26", "normal"), ("r6", "processing", "2028-08-24", "atrasado"),
        ("r7", "invoiced", "2028-08-25", "em_risco"), ("r8", "created", None, "normal"),
        ("r9", "delivered", "2028-08-01", None), ("r10", "canceled", "2028-08-01", None),
        ("r11", "unavailable", "2028-08-01", None),
    ]
    df = spark.createDataFrame([(o, s, e) for o, s, e, _ in casos], "order_id string, status string, estimated string").withColumn(
        "estimated_delivery_date", F.col("estimated").cast("timestamp")
    )
    obtido = {r["order_id"]: r["risco_atraso"] for r in with_risk(df, hoje).collect()}
    for order_id, status, estimada, esperado in casos:
        assert obtido[order_id] == esperado, f"{order_id} ({status}, {estimada}): esperado {esperado!r}, obtido {obtido[order_id]!r}"
    print(f"OK  regra de risco: {len(casos)} casos, data simulada {hoje}, N = {RISK_CONFIG['em_risco_dias']}")
    print("OK  pedidos fechados ficam com risco nulo; despachado só vira atrasado depois de vencer")

In [ ]:
# O QUE FAZ: encerra o notebook depois dos testes das etapas 5.2 a 5.4, antes de qualquer escrita em
#   gold_orders_live (as células seguintes, a execução real, só rodam com teste_leitura = nao).
#
# FUNÇÃO NATIVA:
# - dbutils.notebook.exit(texto): termina o notebook nesse ponto, com sucesso, devolvendo o texto.
if TESTE_LEITURA != "nao":
    dbutils.notebook.exit(f"Teste '{TESTE_LEITURA}' concluído. Nada foi gravado em {LIVE_TABLE}.")

## Execução

Roda o stream de produção: lê da bronze o que chegou desde o último checkpoint (na **primeira** execução, tudo o que já existe: a carga inicial, num único micro-lote), deduplica, aplica o `MERGE` em `gold_orders_live` e registra cada micro-lote em `ops.streaming_batches`. Em seguida recalcula o risco de atraso de todos os pedidos abertos e atualiza a view `vw_pedidos_em_risco` (etapa 5.4).

In [ ]:
# O QUE FAZ: executa o ciclo real em 3 passos:
#   1. o stream (read_events_stream: bronze deduplicada) grava em gold_orders_live pelo foreachBatch + MERGE
#      e registra os micro-lotes em ops.streaming_batches;
#   2. refresh_risk recalcula risco_atraso de todos os pedidos abertos contra a data simulada corrente
#      (também os que não tiveram evento novo) e grava só o que mudou;
#   3. create_risk_view cria ou atualiza a view com a lista de pedidos atrasados e em risco.
#   Depois mostra os micro-lotes desta execução e a contagem por nível de risco.
#   Rodar de novo sem arquivos novos não grava nada novo (o checkpoint sabe até onde já leu).
#
# FUNÇÕES NATIVAS:
# - df.where(condição).orderBy(...): filtra e ordena os registros desta execução.
# - display(df): mostra o resultado em tabela. df.count(): número de linhas.
run_live(read_events_stream(), LIVE_TABLE, BATCHES_TABLE, ORDERS_TABLE, CHECKPOINT_DIR, RUN_ID)

linhas_risco, por_nivel = refresh_risk(LIVE_TABLE, RUN_ID)
create_risk_view(LIVE_TABLE, RISK_VIEW)

display(spark.table(BATCHES_TABLE).where(F.col("run_id") == RUN_ID).orderBy("batch_id"))
print(f"gold_orders_live: {spark.table(LIVE_TABLE).count()} pedidos; data simulada {simulated_today(LIVE_TABLE)}")
print(f"risco recalculado: {linhas_risco} linhas alteradas; por nível: {por_nivel}")